# Test with Dataset from CellTypist

In [1]:
import anndata

# Read data from h5ad files
adata = anndata.io.read_h5ad('../CellTypistDataset/global.h5ad')
print(adata)

AnnData object with n_obs × n_vars = 329762 × 36601
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype'
    obsm: 'X_umap'


In [2]:
# Filter blood data
adata = adata[adata.obs['Organ'] == 'BLD'].copy()
print(adata)

AnnData object with n_obs × n_vars = 27620 × 36601
    obs: 'Organ', 'Donor', 'Chemistry', 'Cell_category', 'Predicted_labels_CellTypist', 'Majority_voting_CellTypist', 'Majority_voting_CellTypist_high', 'Manually_curated_celltype'
    obsm: 'X_umap'


In [3]:
# Prepare Data for training
X = adata.to_df()
gene_names = adata.var_names
y = adata.obs['Manually_curated_celltype']

print(X)
print('---')
print(gene_names)
print('---')
print(y)

                               MIR1302-2HG  FAM138A  OR4F5  AL627309.1  \
Pan_T7980361_AAACCTGAGTGCAAGC          0.0      0.0    0.0         0.0   
Pan_T7980361_AAACCTGCAATACGCT          0.0      0.0    0.0         0.0   
Pan_T7980361_AAACCTGTCCATGCTC          0.0      0.0    0.0         0.0   
Pan_T7980361_AAACCTGTCTGAAAGA          0.0      0.0    0.0         0.0   
Pan_T7980361_AAACCTGTCTGCGTAA          0.0      0.0    0.0         0.0   
...                                    ...      ...    ...         ...   
CZINY-0112_TTGTTTGTCGAACGCC            0.0      0.0    0.0         0.0   
CZINY-0112_TTTAGTCAGAGCTGAC            0.0      0.0    0.0         0.0   
CZINY-0112_TTTAGTCCAGCATTGT            0.0      0.0    0.0         0.0   
CZINY-0112_TTTCACAGTCAAAGAT            0.0      0.0    0.0         0.0   
CZINY-0112_TTTCGATGTGTTTCTT            0.0      0.0    0.0         0.0   

                               AL627309.3  AL627309.2  AL627309.5  AL627309.4  \
Pan_T7980361_AAACCTGAGTGCAAGC 

## Random Forest

In [4]:
from sklearn.model_selection import train_test_split

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 23, test_size = 0.5)

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Random Forest Modell
model = RandomForestClassifier()

# Parameter für das Grid
param_grid = {
    'n_estimators': [50, 100, 200],         # Anzahl der Bäume
    'max_depth': [10, 20, None],            # Maximale Tiefe der Bäume
    'min_samples_split': [2, 5, 10],        # Minimale Stichproben für den Split
    'min_samples_leaf': [1, 2, 4],          # Minimale Stichproben pro Blatt
    'max_features': ['sqrt', 'log2', None]  # Anzahl der Merkmale, die für den Split verwendet werden
}

# GridSearchCV initialisieren
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=1)

# GridSearchCV fitten
grid_search.fit(X_train, y_train)

# Die besten Parameter ausgeben
print("Beste Parameter:", grid_search.best_params_)
print("Beste Genauigkeit:", grid_search.best_score_)

# Modell mit den besten Parametern auf dem Testset evaluieren
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print("Testgenauigkeit mit den besten Parametern:", test_accuracy)

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


KeyboardInterrupt: 